In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
from faker import Faker
import random

# Random seed for reproducibility
random.seed(42)
np.random.seed(42)


#### Clients


In [192]:
# Generate 200 unique clients
num_clients = 200
client_ids = [f'C{1000 + i}' for i in range(num_clients)]


# Possible values
symbols = ['EURUSD','USDJPY','GBPUSD','USDCHF', 'AUDUSD','USDCAD','NZDUSD','EURGBP','EURJPY','GBPJPY', 'BTCUSD']
account_types = ['Standard', 'ECN', 'Micro']
countries = ['MY', 'PH', 'SG', 'TH', 'VN', 'ID']
statuses = ['Approved', 'Rejected', 'Pending']
leverage_options = ['1:100', '1:200', '1:500', '1:1000']
spread_ranges = {
    'EURUSD': (0.8, 1.5),
    'USDJPY': (1.0, 2.0),
    'GBPUSD': (0.7,1.4),
    'USDCHF': (2.0, 5.0), 
    'AUDUSD': (1.0, 2.5),       
    'USDCAD': (1.5, 3.0),
    'NZDUSD': (1.2, 2.8),
    'EURGBP': (0.9, 1.8),
    'EURJPY': (1.0, 2.2),
    'GBPJPY': (1.5, 3.5),
    'BTCUSD': (10.0, 20.0),
}

# generate csv clients
clients = pd.DataFrame({
    'client_id': client_ids,
    'account_type': np.random.choice(account_types, num_clients),
    'country': np.random.choice(countries, num_clients),
    'signup_date': [datetime(2024, 1, 1) + timedelta(days=random.randint(0, 364)) for _ in range(num_clients)],
    'is_active': np.random.choice([True, False], num_clients, p=[0.76, 0.24])
})

clients.head()

,client_id,account_type,country,signup_date,is_active
0,C1000,Standard,MY,2024-11-20,False
1,C1001,Standard,VN,2024-12-22,True
2,C1002,Standard,SG,2024-09-20,True
3,C1003,Standard,PH,2024-04-11,False
4,C1004,Standard,PH,2024-06-26,True


#### Trades

In [202]:
num_trades = 2121
trade_ids = [f'T{1000 + i}' for i in range(num_trades)]

trades = []
for i in range(num_trades):
    client = np.random.choice(client_ids)
    symbol = np.random.choice(symbols)
    open_time = datetime(2024, 1, 1) + timedelta(days=random.randint(0, 180), minutes=random.randint(0, 1440))
    volume = round(np.random.uniform(0.1, 5.0), 2)
    profit = round(np.random.normal(0, 50), 2)
    leverage = np.random.choice(leverage_options)
    spread = round(np.random.uniform(*spread_ranges[symbol]), 2)

    # 10% of trades will be left open (no close_time)
    if random.random() < 0.1:
        close_time = pd.NaT
    # 20% of trades will last 1–5 days
    elif random.random() < 0.2:
        close_time = open_time + timedelta(days=random.randint(1, 5))
    # 70% of trades will last a few hours
    else:
        close_time = open_time + timedelta(minutes=random.randint(1, 320))

    trades.append([
        trade_ids[i], client, symbol,
        open_time, close_time, volume,
        profit, leverage, spread
    ])

# Create DataFrame
trades_df = pd.DataFrame(trades, columns=[
    'trade_id', 'client_id', 'symbol', 'open_time', 'close_time',
    'volume', 'profit', 'leverage', 'spread'
])

# Preview
print(trades_df.head())

  trade_id client_id  symbol           open_time          close_time  volume  \
0    T1000     C1075  GBPJPY 2024-03-18 18:44:00 2024-03-18 19:48:00    3.91   
1    T1001     C1170  AUDUSD 2024-04-21 07:27:00 2024-04-21 07:46:00    3.78   
2    T1002     C1080  GBPUSD 2024-01-06 00:09:00 2024-01-06 00:23:00    1.76   
3    T1003     C1098  USDCAD 2024-02-06 01:46:00 2024-02-06 02:47:00    4.16   
4    T1004     C1194  USDCHF 2024-01-05 06:14:00 2024-01-05 08:57:00    1.46   

   profit leverage  spread  
0  -64.90    1:200    2.73  
1   59.41    1:100    1.64  
2   40.66    1:500    0.72  
3    7.38    1:500    2.47  
4    2.70    1:100    3.06  


In [ ]:
trades.dtypes

trade_id              object
client_id             object
symbol                object
open_time     datetime64[ns]
close_time    datetime64[ns]
volume               float64
profit               float64
leverage              object
spread               float64
dtype: object

#### Transactions 


In [203]:
# Generate Transaction data
num_transactions = 600
transaction_ids = [f'TX{1000 + i}' for i in range(num_transactions)]

transactions = []
for i in range(num_transactions):
    client = np.random.choice(client_ids)
    tx_type = np.random.choice(['Deposit', 'Withdrawal'])
    amount = round(np.random.uniform(50, 10000), 2)
    tx_date = datetime(2024, 1, 1) + timedelta(days=random.randint(0, 180))
    status = np.random.choice(statuses, p=[0.7, 0.2, 0.1])
    transactions.append([transaction_ids[i], client, tx_type, amount, tx_date, status])
    
transactions_df = pd.DataFrame(transactions, columns=['transaction_id', 'client_id', 'tx_type', 'amount', 'tx_date', 'status'])
transactions_df.head()

,transaction_id,client_id,tx_type,amount,tx_date,status
0,TX1000,C1148,Deposit,7208.76,2024-01-07,Rejected
1,TX1001,C1131,Withdrawal,9793.52,2024-03-12,Approved
2,TX1002,C1075,Withdrawal,4928.58,2024-03-08,Rejected
3,TX1003,C1116,Withdrawal,1839.97,2024-05-03,Approved
4,TX1004,C1107,Deposit,211.24,2024-04-09,Approved


In [176]:
transactions_df.dtypes

transaction_id            object
client_id                 object
tx_type                   object
amount                   float64
tx_date           datetime64[ns]
status                    object
dtype: object

#### Account Snapshots

In [214]:
# set date range for account snapshots

# convert dates
trades_df['open_time'] = pd.to_datetime(trades_df['open_time'])
trades_df['close_time'] = pd.to_datetime(trades_df['close_time'])
transactions_df['tx_date'] = pd.to_datetime(transactions_df['tx_date'])

date_range = pd.date_range(start='2024-06-01', end='2024-06-30')

snapshots = []
for date in date_range:
    for client_id in clients['client_id']:
        dep = transactions_df.query("client_id == @client_id and tx_type == 'Deposit' and tx_date <= @date")['amount'].sum()
        wd = transactions_df.query("client_id == @client_id and tx_type == 'Withdrawal' and tx_date <= @date")['amount'].sum()
        closed_pnl = trades_df.query("client_id == @client_id and close_time <= @date")['profit'].sum()
        
        balance = dep - wd + closed_pnl
        
        open_trades = trades_df.query("client_id == @client_id and open_time <= @date and (close_time > @date or close_time.isna())")
        floating_pnl = open_trades['profit'].sum()
        swap = round(np.random.uniform(-10, 5), 2) if not open_trades.empty else 0
        margin_used = round(open_trades['volume'].sum() * 1000 / 100, 2)
        equity = balance + floating_pnl + swap
        free_margin = equity - margin_used
        
        snapshots.append({
            'client_id': client_id,
            'date': date.date(),
            'balance': balance,
            'equity': equity,
            'floating_pnl': floating_pnl,
            'swap': swap,
            'margin_used': margin_used,
            'free_margin': free_margin
        })
        
snapshots_df = pd.DataFrame(snapshots)
snapshots_df.head()
        

,client_id,date,balance,equity,floating_pnl,swap,margin_used,free_margin
0,C1000,2024-06-01,-12468.38,-12468.38,0.00,0.00,0.0,-12468.38
1,C1001,2024-06-01,1528.80,1528.80,0.00,0.00,0.0,1528.80
2,C1002,2024-06-01,7914.82,7832.60,-83.72,1.50,35.1,7797.50
3,C1003,2024-06-01,-4455.64,-4455.64,0.00,0.00,0.0,-4455.64
4,C1004,2024-06-01,7459.12,7331.57,-130.57,3.02,75.1,7256.47


In [ ]:
snapshots_df.iloc[5999]

client_id            C1199
date            2024-06-30
balance           17718.42
equity             17765.1
floating_pnl         49.53
swap                 -2.85
margin_used           21.9
free_margin        17743.2
Name: 5999, dtype: object

#### Client Cashflow

In [180]:
# Generate Client Cashflow
cashflow = transactions_df.copy()

cashflow['description'] = cashflow['tx_type'].map({
    'Deposit': 'Deposit Made',
    'Withdrawal': 'Withdrawal requested'
})

cashflow.head()

,transaction_id,client_id,tx_type,amount,tx_date,status,description
0,TX1000,C1094,Deposit,6140.19,2024-04-11,Rejected,Deposit Made
1,TX1001,C1142,Withdrawal,6895.44,2024-04-09,Approved,Withdrawal requested
2,TX1002,C1108,Withdrawal,1515.37,2024-03-12,Approved,Withdrawal requested
3,TX1003,C1017,Deposit,4240.85,2024-05-06,Rejected,Deposit Made
4,TX1004,C1036,Withdrawal,1959.17,2024-02-02,Approved,Withdrawal requested


#### Trade Analysis

In [217]:
analysis = []

for _, row in trades_df.iterrows():
    duration = (row['close_time'] - row['open_time']).total_seconds() / 60  # minutes
    rr = round(abs(row['profit'] / 10), 2)
    strategy = 'Scalping' if duration <= 30 else 'Day Trading' if duration <= 240 else 'Swing Trading'
    
    analysis.append({
        'trade_id': row['trade_id'],
        'client_id': row['client_id'],
        'symbol': row['symbol'],
        'entry': row['open_time'],
        'exit': row['close_time'],
        'pnl': row['profit'],
        'duration_min': duration,
        'risk_reward': rr,
        'strategy': strategy,
    })
    
trade_analysis_df = pd.DataFrame(analysis)
trade_analysis_df.head()

,trade_id,client_id,symbol,entry,exit,pnl,duration_min,risk_reward,strategy
0,T1000,C1075,GBPJPY,2024-03-18 18:44:00,2024-03-18 19:48:00,-64.90,64.0,6.49,Day Trading
1,T1001,C1170,AUDUSD,2024-04-21 07:27:00,2024-04-21 07:46:00,59.41,19.0,5.94,Scalping
2,T1002,C1080,GBPUSD,2024-01-06 00:09:00,2024-01-06 00:23:00,40.66,14.0,4.07,Scalping
3,T1003,C1098,USDCAD,2024-02-06 01:46:00,2024-02-06 02:47:00,7.38,61.0,0.74,Day Trading
4,T1004,C1194,USDCHF,2024-01-05 06:14:00,2024-01-05 08:57:00,2.70,163.0,0.27,Day Trading


In [182]:
trade_analysis_df['strategy'].value_counts()

strategy
Day Trading      1418
Swing Trading     514
Scalping          189
Name: count, dtype: int64

#### Client Risk Score


In [222]:
scores = []

for client_id in clients['client_id']:
    client_trades = trades_df[trades_df['client_id'] == client_id]
    snap = snapshots_df[snapshots_df['client_id'] == client_id]
    
    avg_dd = snap['floating_pnl'].mean()
    max_dd = snap['floating_pnl'].min()
    avg_margin = snap['margin_used'].mean()
    win_rate = (client_trades['profit'] > 0).mean()
    trade_count = len(client_trades)
    
    risk = (
        "High" if max_dd < -1000 or avg_margin > 700
        else "Medium" if win_rate < 0.5
        else "Low"
    )
    
    scores.append({
        'client_id': client_id,
        'avg_drawdown': avg_dd,
        'max_drawdown': max_dd,
        'avg_margin_used': avg_margin,
        'win_rate_pct': win_rate * 100,
        'trade_count': trade_count,
        'risk_score': risk
    })

client_risk_score_df = pd.DataFrame(scores)
client_risk_score_df.head()


,client_id,avg_drawdown,max_drawdown,avg_margin_used,win_rate_pct,trade_count,risk_score
0,C1000,0.028333,0.00,6.950000,33.333333,9,Medium
1,C1001,0.000000,0.00,0.000000,41.666667,12,Medium
2,C1002,-83.720000,-83.72,35.100000,30.000000,10,Medium
3,C1003,0.000000,0.00,0.000000,37.500000,8,Medium
4,C1004,-130.219333,-130.57,75.646667,53.333333,15,Low
